# 04 - Machine Learning Model Experiments

This notebook experiments with different ML models for fraud detection.

## Objectives
- Prepare data for machine learning
- Train and evaluate anomaly detection models
- Train and evaluate clustering models
- Compare model performance
- Perform feature importance analysis
- Save trained models for production
- Select best model for production

In [ ]:
# Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score
from sklearn.inspection import permutation_importance
import mlflow
import mlflow.sklearn
import joblib
from pathlib import Path
import warnings

warnings.filterwarnings('ignore')

import sys
sys.path.insert(0, '..')

# Database connection (PostgreSQL)
from src.database.connection import (
    load_orders, load_drivers, load_customers, test_connection
)

# Feature engineering
from src.features.aggregations import create_fraud_detection_dataset

# Custom model classes
from src.models.outlier_detection import IsolationForestModel, LOFModel, EnsembleOutlierDetector
from src.models.clustering import KMeansModel, DBSCANModel, FraudClusterAnalyzer
from src.models.risk_scoring import RiskScoringEngine, create_risk_report, get_high_risk_entities

# Settings
from src.config.settings import MLFLOW_TRACKING_URI, MLFLOW_EXPERIMENT_NAME, OUTPUT_DIR

pd.set_option('display.max_columns', None)
%matplotlib inline

In [ ]:
# Setup MLflow
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment(MLFLOW_EXPERIMENT_NAME)

# Ensure model output directory exists
MODEL_DIR = OUTPUT_DIR / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)
print(f"Models will be saved to: {MODEL_DIR}")

In [ ]:
# Test database connection and load data from PostgreSQL
print("Testing database connection...")
if test_connection():
    print("Connection successful!")
else:
    raise ConnectionError("Failed to connect to PostgreSQL database. Please check your credentials.")

# Load data from PostgreSQL
print("\nLoading data from PostgreSQL...")
orders = load_orders()
drivers = load_drivers()
customers = load_customers()

print(f"Loaded {len(orders):,} orders")
print(f"Loaded {len(drivers):,} drivers")
print(f"Loaded {len(customers):,} customers")

## 1. Prepare Dataset for ML

In [ ]:
# Create consolidated dataset for fraud detection
ml_data = create_fraud_detection_dataset(orders, drivers, customers)
print(f"ML Dataset shape: {ml_data.shape}")
print(f"\nColumns available: {list(ml_data.columns)}")
ml_data.head()

In [ ]:
# Select features for modeling
feature_cols = [
    'order_amount', 'items_delivered', 'items_missing', 'total_items',
    'missing_rate', 'is_high_value', 'is_weekend',
    'driver_age', 'trips', 'driver_missing_rate', 'driver_total_orders',
    'customer_age', 'customer_missing_rate', 'customer_total_orders'
]

# Filter columns that exist
available_cols = [col for col in feature_cols if col in ml_data.columns]
print(f"Using {len(available_cols)} features:")
for col in available_cols:
    print(f"  - {col}")

In [ ]:
# Prepare feature matrix
X = ml_data[available_cols].copy()

# Handle missing values
X = X.fillna(X.median())

# Convert boolean to int
for col in X.columns:
    if X[col].dtype == 'bool':
        X[col] = X[col].astype(int)

print(f"Feature matrix shape: {X.shape}")
X.describe()

In [ ]:
# Check for has_missing column for labeling
if 'has_missing' not in ml_data.columns:
    ml_data['has_missing'] = ml_data['items_missing'] > 0

print(f"Orders with missing items: {ml_data['has_missing'].sum():,} ({ml_data['has_missing'].mean()*100:.2f}%)")

## 2. Isolation Forest (Anomaly Detection)

Using the custom `IsolationForestModel` class from `src.models.outlier_detection`.

In [ ]:
# Train Isolation Forest using custom model class
with mlflow.start_run(run_name="isolation_forest_custom"):
    # Model parameters
    contamination = 0.1  # Expected proportion of outliers
    n_estimators = 100
    
    mlflow.log_params({
        "model_type": "IsolationForestModel",
        "contamination": contamination,
        "n_estimators": n_estimators,
        "n_features": len(available_cols),
        "features": str(available_cols)
    })
    
    # Initialize and train model
    iso_forest_model = IsolationForestModel(
        name="isolation_forest_fraud",
        contamination=contamination,
        n_estimators=n_estimators
    )
    iso_forest_model.fit(X)
    
    # Get predictions and scores
    iso_predictions = iso_forest_model.predict(X)
    iso_scores = iso_forest_model.score(X)
    iso_risk_scores = iso_forest_model.get_risk_scores(X)
    
    # Results
    n_anomalies_if = (iso_predictions == -1).sum()
    anomaly_rate_if = n_anomalies_if / len(iso_predictions) * 100
    
    mlflow.log_metrics({
        "n_anomalies": n_anomalies_if,
        "anomaly_rate": anomaly_rate_if,
        "avg_risk_score": iso_risk_scores.mean()
    })
    
    # Save model using custom save method
    iso_model_path = iso_forest_model.save(MODEL_DIR / "isolation_forest_fraud.joblib")
    mlflow.log_artifact(str(iso_model_path))
    
    print(f"Anomalies detected: {n_anomalies_if:,} ({anomaly_rate_if:.2f}%)")
    print(f"Model saved to: {iso_model_path}")

In [ ]:
# Add predictions to data
ml_data['iso_forest_pred'] = iso_predictions
ml_data['iso_forest_score'] = iso_scores
ml_data['iso_forest_risk'] = iso_risk_scores

# Analyze anomalies
anomalies_if = ml_data[ml_data['iso_forest_pred'] == -1]
normal_if = ml_data[ml_data['iso_forest_pred'] == 1]

print("\nAnomaly Characteristics:")
print(f"  Avg missing rate (anomalies): {anomalies_if['missing_rate'].mean():.2f}%")
print(f"  Avg missing rate (normal): {normal_if['missing_rate'].mean():.2f}%")
print(f"  Avg order amount (anomalies): ${anomalies_if['order_amount'].mean():.2f}")
print(f"  Avg order amount (normal): ${normal_if['order_amount'].mean():.2f}")
print(f"  Pct with missing (anomalies): {anomalies_if['has_missing'].mean()*100:.2f}%")
print(f"  Pct with missing (normal): {normal_if['has_missing'].mean()*100:.2f}%")

In [ ]:
# Visualize anomaly scores
fig = px.histogram(
    ml_data,
    x='iso_forest_risk',
    color=ml_data['iso_forest_pred'].map({1: 'Normal', -1: 'Anomaly'}),
    nbins=50,
    title='Isolation Forest Risk Scores Distribution',
    labels={'iso_forest_risk': 'Risk Score (0-100)', 'color': 'Classification'},
    color_discrete_map={'Normal': '#2ecc71', 'Anomaly': '#e74c3c'}
)
fig.update_layout(barmode='overlay')
fig.update_traces(opacity=0.7)
fig.show()

## 3. Local Outlier Factor (LOF)

Using the custom `LOFModel` class from `src.models.outlier_detection`.

In [ ]:
# Train LOF using custom model class
with mlflow.start_run(run_name="lof_custom"):
    n_neighbors = 20
    contamination = 0.1
    
    mlflow.log_params({
        "model_type": "LOFModel",
        "n_neighbors": n_neighbors,
        "contamination": contamination,
        "n_features": len(available_cols)
    })
    
    # Initialize and train model
    lof_model = LOFModel(
        name="lof_fraud",
        n_neighbors=n_neighbors,
        contamination=contamination
    )
    lof_model.fit(X)
    
    # Get predictions and scores
    lof_predictions = lof_model.predict(X)
    lof_scores = lof_model.score(X)
    
    # Results
    n_anomalies_lof = (lof_predictions == -1).sum()
    anomaly_rate_lof = n_anomalies_lof / len(lof_predictions) * 100
    
    mlflow.log_metrics({
        "n_anomalies": n_anomalies_lof,
        "anomaly_rate": anomaly_rate_lof
    })
    
    # Save model
    lof_model_path = lof_model.save(MODEL_DIR / "lof_fraud.joblib")
    mlflow.log_artifact(str(lof_model_path))
    
    print(f"LOF Anomalies detected: {n_anomalies_lof:,} ({anomaly_rate_lof:.2f}%)")
    print(f"Model saved to: {lof_model_path}")

In [ ]:
# Add LOF predictions
ml_data['lof_pred'] = lof_predictions
ml_data['lof_score'] = lof_scores

# Analyze LOF anomalies
anomalies_lof = ml_data[ml_data['lof_pred'] == -1]
normal_lof = ml_data[ml_data['lof_pred'] == 1]

print("\nLOF Anomaly Characteristics:")
print(f"  Avg missing rate (anomalies): {anomalies_lof['missing_rate'].mean():.2f}%")
print(f"  Avg missing rate (normal): {normal_lof['missing_rate'].mean():.2f}%")
print(f"  Pct with missing (anomalies): {anomalies_lof['has_missing'].mean()*100:.2f}%")
print(f"  Pct with missing (normal): {normal_lof['has_missing'].mean()*100:.2f}%")

## 4. K-Means Clustering

Using the custom `KMeansModel` class from `src.models.clustering`.

In [ ]:
# Find optimal k using the static method from KMeansModel
print("Finding optimal number of clusters...")
k_range = range(2, 11)
optimal_k, silhouette_scores = KMeansModel.find_optimal_k(X, k_range)

# Also calculate inertias for elbow method
inertias = []
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X.fillna(X.median()))

from sklearn.cluster import KMeans as SK_KMeans
for k in k_range:
    kmeans_temp = SK_KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans_temp.fit(X_scaled)
    inertias.append(kmeans_temp.inertia_)

# Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(k_range, inertias, 'bo-', linewidth=2, markersize=8)
axes[0].set_xlabel('Number of Clusters (k)', fontsize=12)
axes[0].set_ylabel('Inertia', fontsize=12)
axes[0].set_title('Elbow Method', fontsize=14)
axes[0].grid(True, alpha=0.3)

axes[1].plot(k_range, silhouette_scores, 'ro-', linewidth=2, markersize=8)
axes[1].axvline(x=optimal_k, color='g', linestyle='--', label=f'Optimal k={optimal_k}')
axes[1].set_xlabel('Number of Clusters (k)', fontsize=12)
axes[1].set_ylabel('Silhouette Score', fontsize=12)
axes[1].set_title('Silhouette Score', fontsize=14)
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'plots' / 'kmeans_optimization.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\nOptimal k based on silhouette score: {optimal_k}")
print(f"Best silhouette score: {max(silhouette_scores):.4f}")

In [ ]:
# Train K-Means with optimal k using custom model class
with mlflow.start_run(run_name="kmeans_custom"):
    mlflow.log_params({
        "model_type": "KMeansModel",
        "n_clusters": optimal_k,
        "n_features": len(available_cols)
    })
    
    # Initialize and train model
    kmeans_model = KMeansModel(
        name="kmeans_fraud",
        n_clusters=optimal_k
    )
    kmeans_model.fit(X)
    
    # Get predictions and scores
    kmeans_labels = kmeans_model.predict(X)
    kmeans_silhouette = kmeans_model.get_silhouette_score(X)
    kmeans_distances = kmeans_model.score(X)  # Distance to cluster center
    
    mlflow.log_metrics({
        "silhouette_score": kmeans_silhouette,
        "inertia": kmeans_model.model.inertia_
    })
    
    # Save model
    kmeans_model_path = kmeans_model.save(MODEL_DIR / "kmeans_fraud.joblib")
    mlflow.log_artifact(str(kmeans_model_path))
    
    print(f"K-Means trained with {optimal_k} clusters")
    print(f"Silhouette Score: {kmeans_silhouette:.4f}")
    print(f"Model saved to: {kmeans_model_path}")

In [ ]:
# Add cluster labels
ml_data['cluster'] = kmeans_labels
ml_data['cluster_distance'] = kmeans_distances

# Analyze clusters
cluster_analysis = ml_data.groupby('cluster').agg({
    'order_id': 'count',
    'missing_rate': 'mean',
    'order_amount': 'mean',
    'has_missing': 'mean',
    'items_missing': 'sum'
}).round(2)
cluster_analysis.columns = ['count', 'avg_missing_rate', 'avg_order_amount', 'pct_with_missing', 'total_items_missing']
cluster_analysis['pct_of_total'] = (cluster_analysis['count'] / len(ml_data) * 100).round(2)

# Identify high-risk cluster
high_risk_cluster = cluster_analysis['avg_missing_rate'].idxmax()
cluster_analysis['risk_level'] = cluster_analysis['avg_missing_rate'].apply(
    lambda x: 'High' if x == cluster_analysis['avg_missing_rate'].max() else
              'Medium' if x > cluster_analysis['avg_missing_rate'].mean() else 'Low'
)

print("Cluster Analysis:")
display(cluster_analysis)
print(f"\nHigh-risk cluster: {high_risk_cluster}")

In [ ]:
# Visualize clusters
fig = px.scatter(
    ml_data,
    x='order_amount',
    y='missing_rate',
    color='cluster',
    title='K-Means Clusters: Order Amount vs Missing Rate',
    labels={'order_amount': 'Order Amount ($)', 'missing_rate': 'Missing Rate (%)'},
    color_continuous_scale='viridis'
)
fig.update_traces(marker=dict(size=5, opacity=0.6))
fig.show()

## 5. DBSCAN Clustering

Using the custom `DBSCANModel` class from `src.models.clustering`.

In [ ]:
# Train DBSCAN using custom model class
with mlflow.start_run(run_name="dbscan_custom"):
    eps = 0.5
    min_samples = 10
    
    mlflow.log_params({
        "model_type": "DBSCANModel",
        "eps": eps,
        "min_samples": min_samples,
        "n_features": len(available_cols)
    })
    
    # Initialize and train model
    dbscan_model = DBSCANModel(
        name="dbscan_fraud",
        eps=eps,
        min_samples=min_samples
    )
    dbscan_model.fit(X)
    
    # Get predictions and cluster info
    dbscan_labels = dbscan_model.predict(X)
    cluster_info = dbscan_model.get_cluster_info()
    noise_indices = dbscan_model.get_noise_points(X)
    
    n_clusters_dbscan = cluster_info['n_clusters']
    n_noise_dbscan = cluster_info['n_noise']
    noise_ratio_dbscan = cluster_info['noise_ratio']
    
    mlflow.log_metrics({
        "n_clusters": n_clusters_dbscan,
        "n_noise_points": n_noise_dbscan,
        "noise_ratio": noise_ratio_dbscan
    })
    
    # Save model
    dbscan_model_path = dbscan_model.save(MODEL_DIR / "dbscan_fraud.joblib")
    mlflow.log_artifact(str(dbscan_model_path))
    
    print(f"DBSCAN found {n_clusters_dbscan} clusters")
    print(f"Noise points (potential anomalies): {n_noise_dbscan:,} ({noise_ratio_dbscan*100:.2f}%)")
    print(f"Model saved to: {dbscan_model_path}")

In [ ]:
# Add DBSCAN labels
ml_data['dbscan_cluster'] = dbscan_labels

# Analyze noise points (potential anomalies)
noise_points = ml_data[ml_data['dbscan_cluster'] == -1]
non_noise = ml_data[ml_data['dbscan_cluster'] != -1]

print("\nDBSCAN Noise Point Characteristics:")
print(f"  Avg missing rate (noise): {noise_points['missing_rate'].mean():.2f}%")
print(f"  Avg missing rate (clustered): {non_noise['missing_rate'].mean():.2f}%")
print(f"  Pct with missing items (noise): {noise_points['has_missing'].mean()*100:.2f}%")
print(f"  Pct with missing items (clustered): {non_noise['has_missing'].mean()*100:.2f}%")
print(f"  Avg order amount (noise): ${noise_points['order_amount'].mean():.2f}")
print(f"  Avg order amount (clustered): ${non_noise['order_amount'].mean():.2f}")

## 6. Risk Scoring Engine

Using the `RiskScoringEngine` from `src.models.risk_scoring` to create unified risk scores.

In [ ]:
# Initialize and train Risk Scoring Engine
with mlflow.start_run(run_name="risk_scoring_engine"):
    mlflow.log_params({
        "model_type": "RiskScoringEngine",
        "n_features": len(available_cols)
    })
    
    risk_engine = RiskScoringEngine()
    risk_engine.fit(X)
    
    # Score orders
    order_risk_scores = risk_engine.score_orders(ml_data, X)
    
    # Create risk report
    risk_report = create_risk_report(order_risk_scores)
    
    # Get high-risk entities
    high_risk_orders = get_high_risk_entities(order_risk_scores, threshold=75.0)
    
    mlflow.log_metrics({
        "total_scored": len(order_risk_scores),
        "high_risk_count": len(high_risk_orders),
        "avg_risk_score": risk_report['risk_score'].mean()
    })
    
    print(f"Risk Scoring Complete!")
    print(f"  Total orders scored: {len(order_risk_scores):,}")
    print(f"  High-risk orders (score >= 75): {len(high_risk_orders):,}")
    print(f"  Average risk score: {risk_report['risk_score'].mean():.2f}")
    print(f"\nRisk Category Distribution:")
    print(risk_report['risk_category'].value_counts())

In [ ]:
# Add unified risk scores to ml_data
ml_data['unified_risk_score'] = risk_report['risk_score'].values
ml_data['risk_category'] = risk_report['risk_category'].values

# Show top high-risk orders
print("Top 10 Highest Risk Orders:")
display(risk_report.head(10))

## 7. Model Comparison

In [ ]:
# Compare anomaly detection methods
comparison = pd.DataFrame({
    'Method': ['Isolation Forest', 'LOF', 'DBSCAN (noise)', 'Risk Engine (High+Critical)'],
    'Anomalies Detected': [
        (ml_data['iso_forest_pred'] == -1).sum(),
        (ml_data['lof_pred'] == -1).sum(),
        (ml_data['dbscan_cluster'] == -1).sum(),
        (ml_data['risk_category'].isin(['High', 'Critical'])).sum()
    ],
    'Anomaly Rate (%)': [
        (ml_data['iso_forest_pred'] == -1).mean() * 100,
        (ml_data['lof_pred'] == -1).mean() * 100,
        (ml_data['dbscan_cluster'] == -1).mean() * 100,
        (ml_data['risk_category'].isin(['High', 'Critical'])).mean() * 100
    ],
    'Avg Missing Rate in Anomalies (%)': [
        ml_data[ml_data['iso_forest_pred'] == -1]['missing_rate'].mean(),
        ml_data[ml_data['lof_pred'] == -1]['missing_rate'].mean(),
        ml_data[ml_data['dbscan_cluster'] == -1]['missing_rate'].mean(),
        ml_data[ml_data['risk_category'].isin(['High', 'Critical'])]['missing_rate'].mean()
    ],
    'Pct With Missing Items (%)': [
        ml_data[ml_data['iso_forest_pred'] == -1]['has_missing'].mean() * 100,
        ml_data[ml_data['lof_pred'] == -1]['has_missing'].mean() * 100,
        ml_data[ml_data['dbscan_cluster'] == -1]['has_missing'].mean() * 100,
        ml_data[ml_data['risk_category'].isin(['High', 'Critical'])]['has_missing'].mean() * 100
    ]
})
comparison = comparison.round(2)
print("Model Comparison:")
display(comparison)

In [ ]:
# Consensus anomalies (detected by multiple methods)
ml_data['anomaly_votes'] = (
    (ml_data['iso_forest_pred'] == -1).astype(int) +
    (ml_data['lof_pred'] == -1).astype(int) +
    (ml_data['dbscan_cluster'] == -1).astype(int)
)

# Analyze consensus levels
consensus_summary = ml_data.groupby('anomaly_votes').agg({
    'order_id': 'count',
    'missing_rate': 'mean',
    'has_missing': 'mean',
    'order_amount': 'mean'
}).round(2)
consensus_summary.columns = ['count', 'avg_missing_rate', 'pct_with_missing', 'avg_order_amount']
consensus_summary['pct_with_missing'] = (consensus_summary['pct_with_missing'] * 100).round(2)

print("Consensus Analysis (number of methods flagging as anomaly):")
display(consensus_summary)

consensus_anomalies = ml_data[ml_data['anomaly_votes'] >= 2]
print(f"\nConsensus anomalies (2+ methods agree): {len(consensus_anomalies):,}")
print(f"Avg missing rate: {consensus_anomalies['missing_rate'].mean():.2f}%")
print(f"Pct with missing items: {consensus_anomalies['has_missing'].mean()*100:.2f}%")

## 8. Feature Importance Analysis

In [ ]:
# Calculate feature importance using different approaches
print("Calculating feature importance...\n")

# 1. Correlation with anomaly flags
correlations = {}
for col in available_cols:
    correlations[col] = {
        'iso_forest': X[col].corr(pd.Series(iso_predictions == -1).astype(float)),
        'lof': X[col].corr(pd.Series(lof_predictions == -1).astype(float)),
        'dbscan_noise': X[col].corr(pd.Series(dbscan_labels == -1).astype(float)),
        'has_missing': X[col].corr(ml_data['has_missing'].astype(float))
    }

corr_df = pd.DataFrame(correlations).T
corr_df['avg_importance'] = corr_df.abs().mean(axis=1)
corr_df = corr_df.sort_values('avg_importance', ascending=False)

print("Feature Correlations with Anomaly Flags:")
display(corr_df.round(4))

In [ ]:
# 2. Feature importance based on cluster separation
cluster_centers = kmeans_model.model.cluster_centers_
feature_variance = np.var(cluster_centers, axis=0)

cluster_importance = pd.DataFrame({
    'feature': iso_forest_model.feature_names,
    'cluster_variance': feature_variance
}).sort_values('cluster_variance', ascending=False)

# Normalize to 0-100
cluster_importance['importance_score'] = (
    (cluster_importance['cluster_variance'] - cluster_importance['cluster_variance'].min()) /
    (cluster_importance['cluster_variance'].max() - cluster_importance['cluster_variance'].min()) * 100
).round(2)

print("\nFeature Importance Based on K-Means Cluster Separation:")
display(cluster_importance)

In [ ]:
# 3. Combined feature importance visualization
# Merge importance metrics
importance_df = cluster_importance.copy()
importance_df = importance_df.merge(
    corr_df[['avg_importance']].reset_index().rename(columns={'index': 'feature', 'avg_importance': 'correlation_importance'}),
    on='feature'
)
importance_df['correlation_importance'] = (importance_df['correlation_importance'] * 100).round(2)
importance_df['combined_score'] = ((importance_df['importance_score'] + importance_df['correlation_importance']) / 2).round(2)
importance_df = importance_df.sort_values('combined_score', ascending=True)

# Plot
fig, ax = plt.subplots(figsize=(12, 8))

y_pos = np.arange(len(importance_df))
width = 0.35

bars1 = ax.barh(y_pos - width/2, importance_df['importance_score'], width, label='Cluster Separation', color='#3498db')
bars2 = ax.barh(y_pos + width/2, importance_df['correlation_importance'], width, label='Correlation with Anomalies', color='#e74c3c')

ax.set_yticks(y_pos)
ax.set_yticklabels(importance_df['feature'])
ax.set_xlabel('Importance Score', fontsize=12)
ax.set_title('Feature Importance Analysis', fontsize=14)
ax.legend()
ax.grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'plots' / 'feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nTop 5 Most Important Features:")
top_features = importance_df.sort_values('combined_score', ascending=False).head(5)
for idx, row in top_features.iterrows():
    print(f"  {row['feature']}: {row['combined_score']:.2f}")

## 9. Model Persistence Summary

In [ ]:
# List all saved models
print("Saved Models:")
print("=" * 60)

saved_models = list(MODEL_DIR.glob("*.joblib"))
for model_path in saved_models:
    model_size = model_path.stat().st_size / 1024  # KB
    print(f"  {model_path.name}: {model_size:.2f} KB")

print(f"\nTotal models saved: {len(saved_models)}")
print(f"Models directory: {MODEL_DIR}")

In [ ]:
# Demonstrate loading a saved model
print("Testing model loading...")
loaded_iso_forest = IsolationForestModel.load(MODEL_DIR / "isolation_forest_fraud.joblib")
print(f"  Loaded model: {loaded_iso_forest.name}")
print(f"  Is fitted: {loaded_iso_forest.is_fitted}")
print(f"  Number of features: {len(loaded_iso_forest.feature_names)}")

# Verify predictions match
loaded_predictions = loaded_iso_forest.predict(X)
match_rate = (loaded_predictions == iso_predictions).mean() * 100
print(f"  Prediction match rate: {match_rate:.2f}%")

## 10. Summary and Recommendations

In [ ]:
# Create comprehensive summary table programmatically
summary_data = {
    'Model': ['Isolation Forest', 'LOF', 'K-Means', 'DBSCAN', 'Risk Scoring Engine'],
    'Type': ['Anomaly Detection', 'Anomaly Detection', 'Clustering', 'Density Clustering', 'Ensemble Scoring'],
    'Anomalies/Clusters': [
        f"{n_anomalies_if:,} anomalies",
        f"{n_anomalies_lof:,} anomalies",
        f"{optimal_k} clusters",
        f"{n_clusters_dbscan} clusters + {n_noise_dbscan:,} noise",
        f"{len(high_risk_orders):,} high-risk"
    ],
    'Key Metric': [
        f"{anomaly_rate_if:.2f}% anomaly rate",
        f"{anomaly_rate_lof:.2f}% anomaly rate",
        f"{kmeans_silhouette:.4f} silhouette",
        f"{noise_ratio_dbscan*100:.2f}% noise ratio",
        f"{risk_report['risk_score'].mean():.2f} avg score"
    ],
    'Avg Missing Rate in Flagged': [
        f"{anomalies_if['missing_rate'].mean():.2f}%",
        f"{anomalies_lof['missing_rate'].mean():.2f}%",
        f"{ml_data[ml_data['cluster'] == high_risk_cluster]['missing_rate'].mean():.2f}% (cluster {high_risk_cluster})",
        f"{noise_points['missing_rate'].mean():.2f}%",
        f"{ml_data[ml_data['risk_category'].isin(['High', 'Critical'])]['missing_rate'].mean():.2f}%"
    ],
    'Key Insight': [
        'Effective for detecting global outliers; captures orders with unusual feature combinations',
        'Detects local density anomalies; sensitive to neighborhood structure',
        f'Cluster {high_risk_cluster} has highest missing rate; useful for segmentation',
        'Identifies noise points as potential anomalies; no predefined cluster count',
        'Combines multiple signals into unified risk score; ready for production'
    ]
}

summary_table = pd.DataFrame(summary_data)
print("\n" + "="*100)
print("MODEL PERFORMANCE SUMMARY")
print("="*100)
display(summary_table)

In [ ]:
# Determine recommended model
print("\n" + "="*100)
print("RECOMMENDATION")
print("="*100)

# Calculate effectiveness metrics
baseline_missing_rate = ml_data['missing_rate'].mean()

effectiveness = {
    'Isolation Forest': anomalies_if['missing_rate'].mean() / baseline_missing_rate,
    'LOF': anomalies_lof['missing_rate'].mean() / baseline_missing_rate,
    'DBSCAN': noise_points['missing_rate'].mean() / baseline_missing_rate,
    'Risk Engine': ml_data[ml_data['risk_category'].isin(['High', 'Critical'])]['missing_rate'].mean() / baseline_missing_rate
}

best_model = max(effectiveness, key=effectiveness.get)

print(f"\nBaseline missing rate: {baseline_missing_rate:.2f}%")
print(f"\nModel Effectiveness (flagged missing rate / baseline):")
for model, eff in sorted(effectiveness.items(), key=lambda x: x[1], reverse=True):
    print(f"  {model}: {eff:.2f}x")

print(f"\n*** RECOMMENDED MODEL: {best_model} ***")
print(f"\nRationale:")
print(f"  - Highest effectiveness ratio: {effectiveness[best_model]:.2f}x baseline")
print(f"  - Successfully identifies orders with significantly higher missing rates")
if best_model == 'Risk Engine':
    print(f"  - Provides interpretable risk scores (0-100) and categories")
    print(f"  - Combines multiple anomaly detection signals")
    print(f"  - Ready for production deployment with clear thresholds")

In [ ]:
# Next steps summary
print("\n" + "="*100)
print("NEXT STEPS")
print("="*100)
print("""
1. DEPLOYMENT:
   - Deploy the Risk Scoring Engine for real-time order scoring
   - Set up automated flagging for orders with risk_score >= 75
   - Models saved to: {}

2. MONITORING:
   - Create dashboard to track flagged orders
   - Monitor model drift by comparing predicted vs actual fraud rates
   - Set up alerts for unusual spikes in high-risk orders

3. VALIDATION:
   - Investigate consensus anomalies (flagged by 2+ methods)
   - Conduct manual review of top high-risk orders
   - Collect feedback to create labeled training data

4. IMPROVEMENT:
   - Add temporal features (time since last order, order frequency)
   - Include product category analysis
   - Explore driver-customer pair analysis for collusion detection
   - Retrain models with feedback data for supervised learning
""".format(MODEL_DIR))

In [ ]:
# Save final results
results_path = OUTPUT_DIR / "reports" / "model_experiment_results.csv"
ml_data.to_csv(results_path, index=False)
print(f"\nResults saved to: {results_path}")

# Save summary table
summary_path = OUTPUT_DIR / "reports" / "model_summary.csv"
summary_table.to_csv(summary_path, index=False)
print(f"Summary saved to: {summary_path}")

# Save feature importance
importance_path = OUTPUT_DIR / "reports" / "feature_importance.csv"
importance_df.to_csv(importance_path, index=False)
print(f"Feature importance saved to: {importance_path}")